# Leakage-controlled model development

Recompute HR training/validation selection; the pipeline independently evaluates untouched test data.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [2]:
import json, pandas as pd
from IPython.display import display, Image
results=json.loads((ROOT/'reports/results.json').read_text())

In [3]:
from src.data_cleaning import load_hr
from src.modeling import select_model
from src.evaluation import evaluate
hr=load_hr(ROOT/'data/raw/emp_attrition.csv')
split=pd.read_csv(ROOT/'data/processed/hr_split_assignment.csv')
x=hr.drop(columns=['Attrition','EmployeeNumber','EmployeeCount','Over18','StandardHours','Gender'])
y=hr.Attrition.eq('Yes').astype(int)
train=split.split.eq('train'); val=split.split.eq('validation'); test=split.split.eq('test')
assert not (train & val).any() and not (train & test).any()
model,selection=select_model(x[train],y[train],x[val],y[val],42)
display(selection)

{'selected_model': 'logistic',
 'validation_average_precision': {'logistic': 0.6389160556811722,
  'random_forest': 0.6197111048087534},
 'threshold': 0.32,
 'validation_f1': 0.631578947368421}

In [4]:
probability=model.predict_proba(x[test])[:,1]
display(evaluate(y[test],probability,selection['threshold']))
display(pd.read_csv(ROOT/'data/processed/hr_permutation_importance.csv').head(10))
print('False positives: unnecessary contact or stigma. False negatives: missed support. Educational threshold maximizes validation F1, not business benefit.')

{'n': 294,
 'threshold': 0.32,
 'precision': 0.5,
 'recall': 0.5106382978723404,
 'f1': 0.5052631578947369,
 'roc_auc': 0.785425101214575,
 'average_precision': 0.5718280175265456,
 'brier': 0.10241946147394985,
 'confusion_matrix': [[223, 24], [23, 24]]}

,feature,mean_ap_decrease,std_ap_decrease
0,OverTime,0.206864,0.012970
1,JobSatisfaction,0.084508,0.013843
2,NumCompaniesWorked,0.062953,0.017108
3,YearsSinceLastPromotion,0.052066,0.024071
4,TotalWorkingYears,0.049791,0.025648
5,EnvironmentSatisfaction,0.039653,0.017744
6,DistanceFromHome,0.038571,0.008133
7,JobInvolvement,0.033977,0.011714
8,RelationshipSatisfaction,0.026992,0.008188
9,MonthlyIncome,0.026904,0.008069


False positives: unnecessary contact or stigma. False negatives: missed support. Educational threshold maximizes validation F1, not business benefit.
